# 01 – Explore the MoSPI CPI data (Day 4 project part + Day 5 Pandas)

**Source:** MoSPI eSankhyiki, Consumer Price Index (base year 2024), downloaded 9 Nov 2026.
**Rule:** the raw file is never edited. All changes happen on a copy called `cpi`.

**How to use:** run the cells top to bottom. Cells marked **TODO** are yours: write the code, run it,
and write your finding in a markdown cell below it (observation first, hypothesis second, clearly labelled).
In Colab, upload `cpi_1153.xlsx` and change `RAW_PATH` to `"cpi_1153.xlsx"`.

In [ ]:
import os
import pandas as pd
import numpy as np

RAW_PATH = "data/raw/cpi_1153.xlsx"   # change if your file is elsewhere
raw = pd.read_excel(RAW_PATH)

## Step 1 – First look
Questions to answer from the output: How many rows and columns? What does one row represent?
Which columns are numbers, and which are text?

In [ ]:
print(raw.shape)
print(raw.columns.tolist())
raw.head()

In [ ]:
raw.dtypes

## Step 2 – Missing values
`isna()` marks missing cells as True. `.sum()` counts the Trues per column.

In [ ]:
missing = raw.isna().sum()
missing[missing > 0]

In [ ]:
# Where are the missing 'inflation' values? Count them per year.
raw.groupby("year")["inflation"].apply(lambda s: s.isna().sum())

## Step 3 – Make a working copy (the raw data stays untouched)
Two things to notice below:
- A column called `index` clashes with Pandas' own `.index`, so we rename it `cpi_index`.
- `month` is text, so it sorts alphabetically. We turn it into an *ordered category* so April no longer comes before January.

In [ ]:
cpi = raw.copy()

empty_cols = [c for c in cpi.columns if cpi[c].isna().all()]
print("Entirely empty columns:", empty_cols)
cpi = cpi.drop(columns=empty_cols)

cpi = cpi.rename(columns={"index": "cpi_index"})

month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
cpi["month"] = pd.Categorical(cpi["month"], categories=month_order, ordered=True)

print(cpi.shape)

## Step 4 – Validation checks
An `assert` stops the notebook with an error if the statement is False. Use these to prove your assumptions in code.

In [ ]:
key = ["year", "month", "state", "sector", "division"]
assert cpi.duplicated(key).sum() == 0, "duplicate observations found"
assert len(cpi) == len(raw), "rows changed unexpectedly"
print("Checks passed. Rows per year:")
print(cpi.groupby("year").size())

## Step 5 – Selecting and filtering
Filtering uses boolean masks, like in NumPy. Rules: use `&` (and) / `|` (or), and put **each condition in brackets**.
`and` / `or` will raise an error.

In [ ]:
all_india = cpi[(cpi["state"] == "All India") & (cpi["sector"] == "Combined")]
general = all_india[all_india["division"] == "CPI (General)"]
general[general["year"] == 2026].sort_values("month")[["month", "inflation"]]

## Step 6 – groupby (the code version of a PivotTable)
You built this in Excel on Day 3. Check that your Excel numbers match.

In [ ]:
d2026 = cpi[(cpi["year"] == 2026) & (cpi["sector"] == "Combined")]

by_division = (d2026[d2026["state"] == "All India"]
               .groupby("division")["inflation"]
               .mean()
               .sort_values(ascending=False)
               .round(2))
by_division

In [ ]:
pivot = (d2026[d2026["state"] == "All India"]
         .pivot_table(index="division", columns="month", values="inflation", observed=True))
pivot.round(2)

## Step 7 – Your project questions (TODO)
For each: write the code, then a markdown cell with **Observation** (what the numbers show) and
**Hypothesis** (your guess about why, clearly labelled as not shown by this data).

### Q1. How did Food and beverages inflation change from January to September 2026 (All India, Combined)?
Hint: filter state, sector, year and division, then sort by month. Report the first value, the last value,
the change in **percentage points**, the **percent change**, and the month with the biggest rise.

In [ ]:
# TODO Q1
food = None   # replace None with your filtered DataFrame

### Q2. Which state had the highest average CPI (General) inflation Jan–Sep 2026, and is it driven by rural or urban prices?
Hints: exclude "All India". Use `groupby("state")`. For the second part, filter that state and compare
`sector` values "Rural" and "Urban". Report your definition of "average" in one sentence.

In [ ]:
# TODO Q2a: average General inflation by state (Combined), top 5 and bottom 5

# TODO Q2b: for the top state, compare Rural vs Urban

### Q3. How did Health inflation compare with Transport inflation across Jan–Sep 2026 (All India, Combined)?
Hint: `pivot_table` with `division` filtered to the two divisions, or `unstack`. Compute the gap month by month.

In [ ]:
# TODO Q3

### Q4. Which values look anomalous, and do they change any conclusion?
Hint: `cpi[cpi["year"] == 2026].nlargest(10, "inflation")` shows the largest values.
Look at the neighbouring months of any extreme value before deciding anything. Document it, do not delete it.

In [ ]:
# TODO Q4

## Step 8 – Save a cleaned working copy
Raw stays in `data/raw/`. The cleaned version goes in `data/cleaned/`. Log what you changed in `cleaning_log.md`.

In [ ]:
os.makedirs("data/cleaned", exist_ok=True)
cpi.to_csv("data/cleaned/cpi_working.csv", index=False)
print("Saved", cpi.shape)

## Commit checklist
- `git add notebooks/01_explore_cpi.ipynb data/cleaned/cpi_working.csv cleaning_log.md`
- `git commit -m "Add CPI exploration notebook and cleaned working copy"`
- `git push`